# 2. Scientific Safety Index (SSI) - Pipeline de Análisis de Aditivos

## 📌 Descripción General

Este notebook implementa un análisis científico de seguridad de aditivos alimentarios mediante búsqueda en PubMed.

### 🎯 Objetivo
Clasificar 650+ aditivos en categorías de seguridad basadas en evidencia científica:
- **SEGURO**: Bajo riesgo comprobado
- **PRECAUCIÓN**: Riesgo moderado o Poca evidencia científica robusta
- **EVITAR**: Alto riesgo de seguridad

### ⏱️ Tiempo de Ejecución
- **650 aditivos**: ~5-7 horas
- Incluye checkpoints cada 10 aditivos para poder reanudar si falla

### 📊 Input
`maestro_aditivos_limpio.csv` con columnas: `nombre`, `e_code`

### 💾 Output
`ssi_final_con_semaforo.csv` con columnas: `id`, `e_code`, `nombre`, `traffic_light`, etc.

---

## 🔍 Metodología

### Palabras Clave (16 categorías)

**NEGATIVAS (peligro):**
- `adverse_effects`, `toxicity`, `harm`, `risk`, `danger`
- `cancer`, `mutation`, `allergy`, `neurological`, `hepatic`, `kidney`, `metabolic`

**POSITIVAS (seguridad):**
- `safe`, `approved`, `no_adverse`, `not_harmful`

**CONTEXTO:**
- `in_vitro`, `animal_model`, `human_study`

### Fórmula SSI

$$SSI = \text{peso\_temporal} \times \text{confidence}$$

Donde:
- **peso_temporal** = (ratio_neg_reciente × 0.70) + (ratio_neg_historico × 0.30)
- **confidence** = (√papers / 50) × boost_human_studies
- **boost_human**: 1.0 (≥5 humanos), 0.7 (≥2 animales), 0.4 (<2 in-vitro)

---

## ⚙️ Parámetros Críticos

```python
UMBRAL_DANGEROUS = 0.35      # SSI > 35% = DANGEROUS
UMBRAL_WATCHLIST = 0.20      # SSI 20-35% = WATCHLIST
UMBRAL_MIN_PAPERS = 10       # Mínimo 10 papers para confianza
VENTANA_RECIENTE = 10 años   # Papers publicados en últimos 10 años
VENTANA_HISTORICA = 30 años  # Papers históricos completos
```

In [1]:
import numpy as np
import pandas as pd
import urllib.parse
import requests
import time
import re
import os
from datetime import datetime, timedelta
from typing import Dict, Tuple, List
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

✅ Librerías importadas


## 1. Configuración Global

In [ ]:
# ============================================================================
# CREDENCIALES PUBMED
# ============================================================================

MI_EMAIL = os.getenv("MI_EMAIL")          
API_KEY = os.getenv("API_KEY")              
NOMBRE_HERRAMIENTA = "SSIAnalyzer/4.0"

# ============================================================================
# CONFIGURACIÓN DE CHECKPOINTS
# ============================================================================

CHECKPOINT_INTERVAL = 10
OUTPUT_DIR = "../data"
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "ssi_checkpoint.csv")
FINAL_FILE = os.path.join(OUTPUT_DIR, "ssi_final.csv")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "ssi_progress.txt")
ERROR_FILE = os.path.join(OUTPUT_DIR, "ssi_errors.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# PALABRAS CLAVE
# ============================================================================

PALABRAS_CLAVE = {
    # NEGATIVAS (Peligroso)
    'adverse_effects': '("adverse effect" OR "adverse reaction" OR "adverse event")',
    'toxicity': '(toxic OR toxicity OR toxicological)',
    'harm': '(harmful OR harm OR damage OR damaging)',
    'risk': '(risk OR risky)',
    'danger': '(danger OR dangerous)',
    'cancer': '(cancer OR carcinogen OR carcinogenic)',
    'mutation': '(mutation OR mutagenic OR genotoxic OR "DNA damage")',
    'allergy': '(allergy OR allergic OR hypersensitivity)',
    'neurological': '(neurotoxic OR "neurological effect" OR ADHD)',
    'hepatic': '(hepatotoxic OR "liver damage")',
    'kidney': '(nephrotoxic OR "kidney damage")',
    'metabolic': '(metabolic OR metabolism OR obesity)',
    
    # POSITIVAS (Seguro)
    'safe': '(safe OR safety)',
    'approved': '(approved OR "regulatory approval")',
    'no_adverse': '("no adverse" OR "no effect" OR "not toxic")',
    'not_harmful': '("not harmful" OR "failed to demonstrate" OR "no risk")',
    
    # CONTEXTO
    'in_vitro': '("in vitro" OR "test tube")',
    'animal_model': '(animal OR rat OR mouse)',
    'human_study': '(human OR RCT OR epidemiological)',
}

# ✅ QUERIES BALANCEADAS
# Sin restricción de "food additive" - captura más papers
FOOD_CONTEXT = ''

# Exclusiones REDUCIDAS - solo lo verdaderamente irrelevante médicamente
EXCLUSIONS = 'NOT (pharmaceutical OR medicinal OR "drug therapy" OR disease OR therapy OR hospital OR patient)'

# ============================================================================
# PARÁMETROS
# ============================================================================

VENTANA_RECIENTE_AÑOS = 10
VENTANA_HISTORICA_AÑOS = 30
UMBRAL_DANGEROUS = 0.35      # Aumentado de 0.25
UMBRAL_WATCHLIST = 0.20      # Aumentado de 0.15
UMBRAL_MIN_PAPERS = 10
WEIGHT_IN_VITRO = 0.30
WEIGHT_ANIMAL = 0.50
WEIGHT_HUMAN = 1.0

print(f"✅ CONFIGURACIÓN CARGADA - VERSIÓN 4.0")
print(f"\n📊 CAMBIOS PRINCIPALES:")
print(f"   ✅ Queries SIN restricción 'food additive'")
print(f"   ✅ Exclusiones REDUCIDAS (solo contexto médico)")
print(f"   ✅ Resultado esperado: UNKNOWNS ~20-30%")
print(f"\n⚙️  PARÁMETROS:")
print(f"   UMBRAL_DANGEROUS: {UMBRAL_DANGEROUS}")
print(f"   UMBRAL_WATCHLIST: {UMBRAL_WATCHLIST}")
print(f"   UMBRAL_MIN_PAPERS: {UMBRAL_MIN_PAPERS}")
print(f"   Ponderación: In Vitro={WEIGHT_IN_VITRO}, Animal={WEIGHT_ANIMAL}, Human={WEIGHT_HUMAN}")

✅ CONFIGURACIÓN CARGADA - VERSIÓN 4.0

📊 CAMBIOS PRINCIPALES:
   ✅ Queries SIN restricción 'food additive'
   ✅ Exclusiones REDUCIDAS (solo contexto médico)
   ✅ Resultado esperado: UNKNOWNS ~20-30%

⚙️  PARÁMETROS:
   UMBRAL_DANGEROUS: 0.35
   UMBRAL_WATCHLIST: 0.2
   UMBRAL_MIN_PAPERS: 10
   Ponderación: In Vitro=0.3, Animal=0.5, Human=1.0


## 2. Funciones de Checkpoint

In [3]:
def cargar_progreso_previo() -> Tuple[pd.DataFrame, int]:
    if os.path.exists(CHECKPOINT_FILE):
        try:
            df_previo = pd.read_csv(CHECKPOINT_FILE)
            if len(df_previo) > 0:
                print(f"\n✅ Checkpoint encontrado: {len(df_previo)} aditivos procesados")
                return df_previo, len(df_previo) - 1
        except Exception as e:
            print(f"⚠️  Error leyendo checkpoint: {e}")
            return None, -1
    
    print(f"\n📝 No hay checkpoint previo - comenzando desde 0")
    return None, -1


def guardar_checkpoint(df_resultados: pd.DataFrame, num_aditivos_totales: int):
    try:
        df_resultados.to_csv(CHECKPOINT_FILE, index=False)
        
        with open(PROGRESS_FILE, 'w') as f:
            f.write(f"Procesados: {len(df_resultados)} / {num_aditivos_totales}\n")
            f.write(f"Porcentaje: {(len(df_resultados)/num_aditivos_totales)*100:.1f}%\n")
            f.write(f"Timestamp: {datetime.now().isoformat()}\n")
        
        print(f"  💾 Checkpoint guardado: {len(df_resultados)} aditivos")
    except Exception as e:
        print(f"  ❌ Error: {e}")


def mostrar_progreso(idx_actual: int, total: int):
    porcentaje = (idx_actual / total) * 100
    barra = '█' * int(porcentaje / 2) + '░' * (50 - int(porcentaje / 2))
    print(f"\r  [{barra}] {porcentaje:5.1f}% ({idx_actual}/{total})", end='', flush=True)

print("✅ Funciones checkpoint definidas")

✅ Funciones checkpoint definidas


## 3. Búsqueda PubMed

In [4]:
def query_pubmed_count(
    term: str,
    date_from: str = None,
    date_to: str = None,
    max_retries: int = 3
) -> int:
    
    full_term = term
    
    if date_from or date_to:
        date_filter = ""
        if date_from:
            date_filter += f"{date_from}[PDAT] : "
        if date_to:
            date_filter += f"{date_to}[PDAT]"
        elif date_from:
            date_filter += f"9999[PDAT]"
        
        if date_filter:
            full_term += f" AND {date_filter}"
    
    term_encoded = urllib.parse.quote_plus(full_term, safe=':"[]/()')
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={term_encoded}&retmode=json&retmax=0&tool={NOMBRE_HERRAMIENTA}&email={MI_EMAIL}&api_key={API_KEY}"
    
    for intento in range(max_retries):
        try:
            r = requests.get(url, timeout=10)
            
            if r.status_code == 429:
                time.sleep(10 * (intento + 1))
                continue
            
            if r.status_code in (500, 502, 503, 504):
                time.sleep(5 * (intento + 1))
                continue
            
            r.raise_for_status()
            data = r.json()
            
            if "esearchresult" in data and "count" in data["esearchresult"]:
                return int(data["esearchresult"]["count"])
            return 0
        
        except Exception as e:
            if intento == max_retries - 1:
                return 0
            time.sleep(2)
    
    return 0


def obtener_counts_aditivo(nombre: str, e_code: str) -> Dict[str, int]:
    """
    Búsqueda BALANCEADA: Sin restricción "food additive", exclusiones reducidas
    """
    
    # Query balanceada
    if FOOD_CONTEXT:
        query_base = f'((\"{nombre}\" OR \"{e_code}\") AND {FOOD_CONTEXT} {EXCLUSIONS})'
    else:
        query_base = f'((\"{nombre}\" OR \"{e_code}\") {EXCLUSIONS})'
    
    current_year = datetime.now().year
    year_reciente = current_year - VENTANA_RECIENTE_AÑOS
    year_historico = current_year - VENTANA_HISTORICA_AÑOS
    
    counts = {
        'nombre': nombre,
        'e_code': e_code,
        'total': 0,
        'total_reciente': 0,
        'total_historico': 0,
    }
    
    counts['total'] = query_pubmed_count(query_base)
    counts['total_reciente'] = query_pubmed_count(query_base, date_from=str(year_reciente))
    counts['total_historico'] = query_pubmed_count(query_base, date_from=str(year_historico), date_to=str(year_reciente-1))
    
    time.sleep(0.15)
    
    for keyword, query_term in PALABRAS_CLAVE.items():
        full_query = f"{query_base} AND {query_term}"
        counts[f'{keyword}_total'] = query_pubmed_count(full_query)
        counts[f'{keyword}_reciente'] = query_pubmed_count(full_query, date_from=str(year_reciente))
        time.sleep(0.15)
    
    return counts

print("✅ Funciones búsqueda definidas")

✅ Funciones búsqueda definidas


## 4. Filtros Anti-Ruido

In [5]:
def aplicar_filtro_negaciones(counts: Dict[str, int]) -> Dict[str, int]:
    counts_adjusted = counts.copy()
    
    not_harmful_total = counts.get('not_harmful_total', 0)
    adverse_total = counts.get('adverse_effects_total', 0)
    
    if not_harmful_total > 0 and adverse_total > 0:
        reduction_factor = min(0.4, not_harmful_total / adverse_total)
        counts_adjusted['adverse_effects_total'] = max(
            0,
            counts['adverse_effects_total'] - int(counts['adverse_effects_total'] * reduction_factor)
        )
        counts_adjusted['adverse_effects_reciente'] = max(
            0,
            counts['adverse_effects_reciente'] - int(counts['adverse_effects_reciente'] * reduction_factor)
        )
    
    return counts_adjusted


def aplicar_ponderacion_tipo_estudio(counts: Dict[str, int]) -> Dict[str, float]:
    papers_negative = (
        counts.get('adverse_effects_total', 0) +
        counts.get('toxicity_total', 0) +
        counts.get('cancer_total', 0) +
        counts.get('mutation_total', 0) +
        counts.get('neurological_total', 0) +
        counts.get('hepatic_total', 0) +
        counts.get('kidney_total', 0) +
        counts.get('metabolic_total', 0)
    )
    
    in_vitro_mentions = counts.get('in_vitro_total', 0)
    animal_mentions = counts.get('animal_model_total', 0)
    human_mentions = counts.get('human_study_total', 0)
    total_mentions = in_vitro_mentions + animal_mentions + human_mentions
    
    if total_mentions > 0:
        pct_vitro = in_vitro_mentions / total_mentions
        pct_animal = animal_mentions / total_mentions
        pct_human = human_mentions / total_mentions
    else:
        pct_vitro, pct_animal, pct_human = 0.4, 0.4, 0.2
    
    if papers_negative == 0:
        papers_negative_weighted = 0
    else:
        average_weight = (
            pct_vitro * WEIGHT_IN_VITRO +
            pct_animal * WEIGHT_ANIMAL +
            pct_human * WEIGHT_HUMAN
        )
        papers_negative_weighted = papers_negative * average_weight
    
    return {
        'papers_negative_weighted': papers_negative_weighted,
        'pct_human': pct_human,
    }

print("✅ Filtros definidos")

✅ Filtros definidos


## 5. Cálculo SSI

In [6]:
def calcular_ssi_robusto(counts: Dict[str, int]) -> Dict:
    nombre = counts['nombre']
    total = counts['total']
    total_reciente = counts['total_reciente']
    total_historico = counts['total_historico']
    
    if total == 0:
        return {
            'nombre': nombre,
            'e_code': counts.get('e_code', ''),
            'status': 'UNKNOWN',
            'ssi': 0,
            'confidence': 0,
            'total_papers': 0,
            'papers_negative': 0,
            'ratio_negative': 0,
            'pct_human_studies': 0,
            'reason': 'No scientific literature found'
        }
    
    counts = aplicar_filtro_negaciones(counts)
    weighted = aplicar_ponderacion_tipo_estudio(counts)
    papers_negative_weighted = weighted['papers_negative_weighted']
    pct_human = weighted['pct_human']
    
    papers_neg_reciente = (
        counts.get('adverse_effects_reciente', 0) +
        counts.get('toxicity_reciente', 0) +
        counts.get('cancer_reciente', 0) +
        counts.get('mutation_reciente', 0) +
        counts.get('neurological_reciente', 0)
    )
    
    if total_reciente > 0:
        ratio_neg_reciente = papers_neg_reciente / total_reciente
    else:
        ratio_neg_reciente = 0
    
    if total_historico > 0:
        papers_neg_historico = papers_negative_weighted - papers_neg_reciente
        ratio_neg_historico = papers_neg_historico / total_historico
    else:
        ratio_neg_historico = 0
    
    peso_temporal = ratio_neg_reciente * 0.70 + ratio_neg_historico * 0.30
    
    confidence_base = min(1.0, np.sqrt(total) / 50) if total > 0 else 0
    n_human_equiv = total * pct_human
    
    if n_human_equiv >= 5:
        confidence_boost = 1.0
    elif n_human_equiv >= 2:
        confidence_boost = 0.7
    else:
        confidence_boost = 0.4
    
    confidence_factor = confidence_base * confidence_boost
    ssi = peso_temporal * confidence_factor
    
    if total < UMBRAL_MIN_PAPERS and ssi < UMBRAL_WATCHLIST:
        status = 'UNKNOWN'
        reason = f'Insufficient research ({total} papers)'
    elif ssi > UMBRAL_DANGEROUS:
        status = 'DANGEROUS'
        reason = f'High negative evidence'
    elif ssi > UMBRAL_WATCHLIST:
        status = 'WATCHLIST'
        reason = f'Moderate concerns (SSI: {ssi*100:.1f}%)'
    else:
        status = 'SAFE'
        reason = f'Low negative evidence'
    
    return {
        'nombre': nombre,
        'e_code': counts.get('e_code', ''),
        'status': status,
        'ssi': round(ssi, 4),
        'confidence': round(confidence_factor, 2),
        'total_papers': total,
        'papers_negative': int(papers_negative_weighted),
        'ratio_negative': round(papers_negative_weighted / total if total > 0 else 0, 3),
        'pct_human_studies': round(pct_human * 100, 0),
        'reason': reason
    }

print("✅ SSI definido")

✅ SSI definido


## 6. Pipeline Principal

In [7]:
def procesar_aditivos_con_checkpoints(df_aditivos: pd.DataFrame) -> pd.DataFrame:
    df = df_aditivos.copy()
    if 'codigo_e' in df.columns and 'e_code' not in df.columns:
        df.rename(columns={'codigo_e': 'e_code'}, inplace=True)
    
    df_previo, ultimo_idx = cargar_progreso_previo()
    
    if df_previo is not None and len(df_previo) > 0:
        resultados = df_previo.to_dict('records')
        inicio_idx = ultimo_idx + 1
    else:
        resultados = []
        inicio_idx = 0
    
    print(f"\n{'='*80}")
    print(f"PROCESANDO ADITIVOS - VERSIÓN 4.0 BALANCEADA")
    print(f"{'='*80}")
    print(f"Total: {len(df)} | Reanudando desde: {inicio_idx}")
    print(f"Checkpoint cada: {CHECKPOINT_INTERVAL} aditivos\n")
    
    df_trabajo = df.iloc[inicio_idx:]
    errores_log = []
    
    try:
        for i, (idx, row) in enumerate(df_trabajo.iterrows(), start=1):
            nombre = str(row['nombre']).strip()
            e_code = str(row['e_code']).strip() if 'e_code' in row and pd.notna(row['e_code']) else f"E{idx}"
            
            try:
                counts = obtener_counts_aditivo(nombre, e_code)
                counts['e_code'] = e_code
                resultado = calcular_ssi_robusto(counts)
                resultados.append(resultado)
                
                mostrar_progreso(inicio_idx + i, len(df))
                
                if i % CHECKPOINT_INTERVAL == 0:
                    df_temp = pd.DataFrame(resultados)
                    guardar_checkpoint(df_temp, len(df))
                    print(f"\n  ✅ Checkpoint {i//CHECKPOINT_INTERVAL}")
            
            except ZeroDivisionError as e:
                print(f"\n  ⚠️  Error: {nombre}")
                errores_log.append({
                    'tipo': 'ZeroDivisionError',
                    'nombre': nombre,
                    'e_code': e_code,
                    'mensaje': str(e)
                })
                resultado_error = {
                    'nombre': nombre,
                    'e_code': e_code,
                    'status': 'ERROR',
                    'ssi': 0,
                    'confidence': 0,
                    'total_papers': 0,
                    'papers_negative': 0,
                    'ratio_negative': 0,
                    'pct_human_studies': 0,
                    'reason': f'Error: {str(e)}'
                }
                resultados.append(resultado_error)
            
            except Exception as e:
                print(f"\n  ❌ Error: {nombre} - {type(e).__name__}")
                errores_log.append({
                    'tipo': type(e).__name__,
                    'nombre': nombre,
                    'e_code': e_code,
                    'mensaje': str(e)
                })
                resultado_error = {
                    'nombre': nombre,
                    'e_code': e_code,
                    'status': 'ERROR',
                    'ssi': 0,
                    'confidence': 0,
                    'total_papers': 0,
                    'papers_negative': 0,
                    'ratio_negative': 0,
                    'pct_human_studies': 0,
                    'reason': f'{type(e).__name__}: {str(e)}'
                }
                resultados.append(resultado_error)
    
    except KeyboardInterrupt:
        print(f"\n\n⚠️  INTERRUPCIÓN - Guardando...")
    
    finally:
        if resultados:
            df_final = pd.DataFrame(resultados)
            df_final.to_csv(FINAL_FILE, index=False)
            
            print(f"\n\n{'='*80}")
            print(f"✅ COMPLETADO: {len(df_final)} aditivos procesados")
            print(f"{'='*80}")
            print(f"\nGuardado en: {FINAL_FILE}")
            
            if len(errores_log) > 0:
                print(f"\n⚠️  Errores: {len(errores_log)}")
                df_errores = pd.DataFrame(errores_log)
                df_errores.to_csv(ERROR_FILE, index=False)
            
            print(f"\n📊 DISTRIBUCIÓN:")
            for status, count in df_final['status'].value_counts().items():
                pct = (count / len(df_final)) * 100
                print(f"  {status}: {count} ({pct:.1f}%)")
            
            print(f"\n📈 ESTADÍSTICAS SSI:")
            ssi_data = df_final[df_final['status'] != 'ERROR']['ssi']
            if len(ssi_data) > 0:
                print(f"  Media: {ssi_data.mean():.4f}")
                print(f"  Mediana: {ssi_data.median():.4f}")
                print(f"  Rango: {ssi_data.min():.4f} - {ssi_data.max():.4f}")
            
            return df_final
    
    return None

print("✅ Pipeline definido")

✅ Pipeline definido


## 7. Prueba / Ejecución

In [8]:
# Crear dataset de prueba
df_lista_aditivos = pd.read_csv('../data/maestro_aditivos_limpio.csv')
df_ssi = procesar_aditivos_con_checkpoints(df_lista_aditivos)

print("\n" + "="*80)
print("INFORMACIÓN IMPORTANTE")
print("="*80)
print(f"""
✅ NOTEBOOK VERSIÓN 4.0 BALANCEADA

📊 CAMBIOS:
   - Queries SIN restricción "food additive"
   - Exclusiones REDUCIDAS (solo médicas)
   - Resultado esperado: UNKNOWNS ~20-30%

⚙️  ANTES DE EJECUTAR:
   1. Configura MI_EMAIL y API_KEY (celda 1)
   2. Descomentar celda siguiente
   3. Ejecutar

⏱️  TIEMPO ESTIMADO:
   - 650 aditivos: 5-7 horas
   - Checkpoints cada 10 aditivos
   - Auto-resume si falla

📁 ARCHIVOS GENERADOS:
   - {CHECKPOINT_FILE}
   - {FINAL_FILE}
   - {PROGRESS_FILE}
""")


📝 No hay checkpoint previo - comenzando desde 0

PROCESANDO ADITIVOS - VERSIÓN 4.0 BALANCEADA
Total: 651 | Reanudando desde: 0
Checkpoint cada: 10 aditivos

  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   1.5% (10/651)  💾 Checkpoint guardado: 10 aditivos

  ✅ Checkpoint 1
  [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   3.1% (20/651)  💾 Checkpoint guardado: 20 aditivos

  ✅ Checkpoint 2
  [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   4.6% (30/651)  💾 Checkpoint guardado: 30 aditivos

  ✅ Checkpoint 3
  [███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   6.1% (40/651)  💾 Checkpoint guardado: 40 aditivos

  ✅ Checkpoint 4
  [███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   7.7% (50/651)  💾 Checkpoint guardado: 50 aditivos

  ✅ Checkpoint 5
  [████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   9.2% (60/651)  💾 Checkpoint guardado: 60 aditivos

  ✅ Checkpoint 6
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  10.8% (70/651)  💾 Checkpoint guardad

## 8. Ver Resultado Final

In [9]:
if os.path.exists(FINAL_FILE):
    df_final = pd.read_csv(FINAL_FILE)
    print(f"\n✅ Resultado final: {len(df_final)} aditivos\n")
    print(f"📊 Distribución:")
    for status, count in df_final['status'].value_counts().items():
        pct = (count / len(df_final)) * 100
        print(f"  {status}: {count} ({pct:.1f}%)")
else:
    print("\n📝 Resultado final no disponible")


✅ Resultado final: 651 aditivos

📊 Distribución:
  SAFE: 427 (65.6%)
  DANGEROUS: 88 (13.5%)
  WATCHLIST: 70 (10.8%)
  UNKNOWN: 66 (10.1%)


## 9. Parcheo prosvisional 

La clasificación ha resultado ser muy estricata y aditivos que deberían estar en el grupo de safe han acabado cayendo en los otros grupos. Por ello se ha añadido una nueva condición, donde si el aditivo pasa a un grupo inferior por poco SSI, pero al mismo tiempo es aprovado por la EFSA, se queda en el grupo safe. Consiguiendo así una clasificación mucho mas balanceada.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd

# Listas EFSA de Perplexity
EFSA_APPROVED = {
    'E100', 'E101', 'E102', 'E104', 'E110', 'E120', 'E122', 'E124', 'E127', 'E128', 'E129',
    'E131', 'E132', 'E133', 'E140', 'E141', 'E142', 'E150a', 'E150b', 'E150c', 'E150d',
    'E160a', 'E160b', 'E160c', 'E160d', 'E160e', 'E160f', 'E161a', 'E161b', 'E161c', 'E161d',
    'E161e', 'E161f', 'E161g', 'E162', 'E163', 'E170', 'E172', 'E173', 'E174', 'E175',
    'E200', 'E201', 'E202', 'E203', 'E210', 'E211', 'E212', 'E213', 'E214', 'E215', 'E216',
    'E217', 'E218', 'E219', 'E220', 'E221', 'E222', 'E223', 'E224', 'E226', 'E227', 'E230',
    'E231', 'E232', 'E233', 'E234', 'E235', 'E236', 'E237', 'E238', 'E239', 'E242', 'E249',
    'E250', 'E251', 'E252', 'E260', 'E261', 'E262', 'E263', 'E270', 'E280', 'E281', 'E282',
    'E283', 'E300', 'E301', 'E302', 'E303', 'E304', 'E305', 'E306', 'E307', 'E308', 'E309',
    'E310', 'E311', 'E312', 'E315', 'E316', 'E319', 'E320', 'E321', 'E322', 'E325', 'E326',
    'E327', 'E330', 'E331', 'E332', 'E333', 'E334', 'E335', 'E336', 'E337', 'E338', 'E339',
    'E340', 'E341', 'E343', 'E350', 'E351', 'E352', 'E353', 'E355', 'E356', 'E357', 'E363',
    'E365', 'E366', 'E370', 'E375', 'E380', 'E385', 'E391', 'E392', 'E400', 'E401', 'E402',
    'E403', 'E404', 'E405', 'E406', 'E407', 'E407a', 'E410', 'E412', 'E413', 'E414', 'E415',
    'E416', 'E417', 'E418', 'E419', 'E420', 'E421', 'E422', 'E425', 'E426', 'E427', 'E428',
    'E429', 'E430', 'E431', 'E432', 'E433', 'E434', 'E435', 'E436', 'E440', 'E441', 'E442',
    'E443', 'E444', 'E445', 'E450', 'E451', 'E452', 'E459', 'E460', 'E461', 'E462', 'E463',
    'E464', 'E465', 'E466', 'E468', 'E469', 'E470a', 'E470b', 'E471', 'E472a', 'E472b',
    'E472c', 'E472d', 'E472e', 'E472f', 'E473', 'E474', 'E475', 'E476', 'E477', 'E478',
    'E479b', 'E481', 'E482', 'E483', 'E484', 'E491', 'E492', 'E493', 'E494', 'E495',
    'E500', 'E501', 'E503', 'E504', 'E507', 'E508', 'E509', 'E511', 'E512', 'E513', 'E514',
    'E515', 'E516', 'E517', 'E520', 'E521', 'E522', 'E523', 'E524', 'E525', 'E526', 'E527',
    'E528', 'E529', 'E530', 'E535', 'E536', 'E540', 'E541', 'E551', 'E552', 'E553a', 'E553b',
    'E554', 'E555', 'E556', 'E559', 'E570', 'E574', 'E575', 'E576', 'E577', 'E578', 'E579',
    'E585', 'E586', 'E620', 'E621', 'E622', 'E623', 'E624', 'E625', 'E626', 'E627', 'E628',
    'E629', 'E630', 'E631', 'E632', 'E633', 'E635', 'E640', 'E650', 'E660', 'E661', 'E663',
    'E900', 'E901', 'E902', 'E903', 'E904', 'E912', 'E914', 'E920', 'E927b', 'E938', 'E939',
    'E941', 'E942', 'E948', 'E949', 'E950', 'E951', 'E952', 'E953', 'E954', 'E955', 'E956',
    'E957', 'E959', 'E960', 'E961', 'E962', 'E965', 'E966', 'E967', 'E968', 'E1000', 'E1001',
    'E1103', 'E1105', 'E1200', 'E1201', 'E1202', 'E1203', 'E1204', 'E1205', 'E1206', 'E1207',
    'E1208', 'E1209', 'E1210', 'E1212', 'E1214', 'E1215', 'E1216', 'E1217', 'E1220', 'E1221',
    'E1404', 'E1410', 'E1412', 'E1413', 'E1414', 'E1420', 'E1422', 'E1440', 'E1442', 'E1450', 'E1451'
}

EFSA_RESTRICTED = {
    'E171', 'E173', 'E174', 'E175', 'E200', 'E202',
    'E249', 'E250', 'E251', 'E252', 'E310', 'E311', 'E312',
    'E338', 'E339', 'E340', 'E341', 'E343',
    'E420', 'E421', 'E950', 'E951', 'E952', 'E954', 'E955', 'E960', 'E961', 'E962', 'E968'
}

EFSA_WITHDRAWN = {
    'E203', 'E216', 'E217', 'E240', 'E924', 'E924b', 'E925', 'E926'
}

def main():
    print("\n" + "="*80)
    print("QUICK PATCH: SSI v4.0 + EFSA Status")
    print("="*80)
    
    # Cargar
    print("\n📖 Cargando ssi_final.csv...")
    df = pd.read_csv('../data/ssi_final.csv')
    print(f"   ✅ {len(df)} aditivos")
    
    # Agregar EFSA status
    print("\n🔄 Agregando EFSA status...")
    
    def get_efsa_status(e_code):
        if e_code in EFSA_WITHDRAWN:
            return 'WITHDRAWN'
        elif e_code in EFSA_RESTRICTED:
            return 'RESTRICTED'
        elif e_code in EFSA_APPROVED:
            return 'APPROVED'
        else:
            return 'APROVED'
    
    df['efsa_status'] = df['e_code'].apply(get_efsa_status)
    
    # Aplicar parcheo
    print("\n⚙️  Aplicando parcheo EFSA...")
    
    overrides = 0
    for idx, row in df.iterrows():
        e_code = row['e_code']
        status_anterior = row['status']
        
        # Si está WITHDRAWN, cambiar a WITHDRAWN
        if e_code in EFSA_WITHDRAWN:
            df.loc[idx, 'status'] = 'WITHDRAWN'
            overrides += 1
        
        # Si está RESTRICTED pero no es, cambiar a RESTRICTED
        elif e_code in EFSA_RESTRICTED and status_anterior not in ['RESTRICTED', 'WITHDRAWN']:
            df.loc[idx, 'status'] = 'RESTRICTED'
            overrides += 1
        
        # Si está APPROVED y dice DANGEROUS, cambiar a SAFE
        elif e_code in EFSA_APPROVED and status_anterior in ['DANGEROUS']:
            df.loc[idx, 'status'] = 'SAFE'
            overrides += 1
    
    print(f"   ✅ {overrides} aditivos parchados")
    
    # Guardar
    output_file = '../data/ssi_final_efsa_patched.csv'
    df.to_csv(output_file, index=False)
    print(f"\n💾 Guardado: {output_file}")
    
    # Estadísticas
    print(f"\n📊 DISTRIBUCIÓN FINAL:")
    for status in ['SAFE', 'DANGEROUS', 'WATCHLIST', 'RESTRICTED', 'WITHDRAWN']:
        count = (df['status'] == status).sum()
        if count > 0:
            pct = (count / len(df)) * 100
            print(f"   {status:12s}: {count:3d} ({pct:5.1f}%)")
    
    print(f"\n📊 EFSA STATUS:")
    for status in ['APPROVED', 'RESTRICTED', 'WITHDRAWN']:
        count = (df['efsa_status'] == status).sum()
        if count > 0:
            pct = (count / len(df)) * 100
            print(f"   {status:12s}: {count:3d} ({pct:5.1f}%)")
    
    print(f"\n✅ COMPLETADO")

if __name__ == '__main__':
    main()

## 10. Clasificación final 

Con toda la información se clasifican los aditivos en 3 grandes grupos: Seguros, Evitables y Precaución. Según en nivel de evidencia y su calidad, siendo "Precaución" un grupo intermedio entre Seguros y Evitables debido a la falta de evidencia científica robusta.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd

def get_traffic_light(row):
    """
    Semáforo de 3 colores: Combinación EFSA_STATUS + SSI
    
    🔴 EVITAR:    Withdrawn o SSI > 0.35 (evitar)
    🟡 PRECAUCION: Restricted o SSI 0.20-0.35 (precaución)
    🟢 SEGURO:    Approved + SSI < 0.20 (seguro)
    """
    
    efsa = row['efsa_status']
    ssi = row['ssi']
    
    # ROJO: Retirados por EFSA
    if efsa == 'WITHDRAWN':
        return 'EVITAR'
    
    # ROJO: SSI muy alto (evidencia negativa fuerte)
    if ssi > 0.35:
        return 'EVITAR'
    
    # AMARILLO: Restringidos por EFSA
    if efsa == 'RESTRICTED':
        return 'PRECAUCION'
    
    # AMARILLO: SSI moderado (evidencia negativa moderada)
    if ssi > 0.20:
        return 'PRECAUCION'
    
    # VERDE: Aprobados + SSI bajo
    if efsa == 'APPROVED' and ssi <= 0.20:
        return 'SEGURO'
    
    # Default VERDE para casos edge
    return 'SEGURO'


def main():
    print("\n" + "="*80)
    print("SEMÁFORO DE 3 COLORES: ADITIVOS")
    print("="*80)
    
    # Cargar
    print("\n📖 Cargando ssi_final_efsa_patched.csv...")
    df = pd.read_csv('../data/dataset_final_codigos_efsa.csv')
    print(f"   ✅ {len(df)} aditivos")
    
    # Generar semáforo
    print("\n🎨 Generando semáforo...")
    df['traffic_light'] = df.apply(get_traffic_light, axis=1)
    
    # Guardar
    output_file = '../data/ssi_final_con_semaforo.csv'
    df.to_csv(output_file, index=False)
    print(f"\n💾 Guardado: {output_file}")
    
    # Estadísticas
    print(f"\n📊 DISTRIBUCIÓN SEMÁFORO:")
    colors = df['traffic_light'].value_counts()
    
    for color in ['SEGURO', 'PRECAUCION', 'EVITAR']:
        count = colors.get(color, 0)
        pct = (count / len(df)) * 100
        print(f"   {color:20s}: {count:3d} ({pct:5.1f}%)")
    
    # Ejemplos por color
    print(f"\n📋 EJEMPLOS:")
    
    print(f"\n   🟢 VERDE (seguros):")
    verde = df[df['traffic_light'] == '🟢 VERDE'].head(3)
    for idx, row in verde.iterrows():
        print(f"      {row['e_code']:8s} {row['nombre']:30s} SSI={row['ssi']:.4f}")
    
    print(f"\n   🟡 AMARILLO (precaución):")
    amarillo = df[df['traffic_light'] == '🟡 AMARILLO'].head(3)
    for idx, row in amarillo.iterrows():
        print(f"      {row['e_code']:8s} {row['nombre']:30s} SSI={row['ssi']:.4f} ({row['efsa_status']})")
    
    print(f"\n   🔴 ROJO (evitar):")
    rojo = df[df['traffic_light'] == '🔴 ROJO'].head(3)
    for idx, row in rojo.iterrows():
        print(f"      {row['e_code']:8s} {row['nombre']:30s} SSI={row['ssi']:.4f} ({row['efsa_status']})")
    
    print(f"\n✅ COMPLETADO")
    print(f"\nUsalo en tu pipeline:")
    print(f"   df = pd.read_csv('{output_file}')")
    print(f"   color = df[df['e_code'] == 'E110']['traffic_light'].values[0]")


if __name__ == '__main__':
    main()